Build a reproducible scikit-learn regression pipeline to predict items_sold at a retail store.
load data

In [1]:
import pandas as pd
df = pd.read_csv("C:/pramod/New start of career/lectures/Assignment 4/part_a/q3_retail_promotions.csv")

In [2]:
df

,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249
3,2022-01-02,17,small,urban,free_gift,1,0,7,259
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277
...,...,...,...,...,...,...,...,...,...
1195,2024-12-28,39,large,urban,bogo,1,0,3,384
1196,2024-12-28,44,medium,urban,category_offer,1,0,9,371
1197,2024-12-29,47,large,semi-urban,bogo,1,0,7,367
1198,2024-12-31,25,small,urban,bogo,0,0,6,321


In [3]:
df.shape

(1200, 9)

In [4]:
df.head()

,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249
3,2022-01-02,17,small,urban,free_gift,1,0,7,259
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   transaction_date     1200 non-null   str  
 1   store_id             1200 non-null   int64
 2   store_size           1200 non-null   str  
 3   location_type        1200 non-null   str  
 4   promotion_type       1200 non-null   str  
 5   is_weekend           1200 non-null   int64
 6   is_festival          1200 non-null   int64
 7   competition_density  1200 non-null   int64
 8   items_sold           1200 non-null   int64
dtypes: int64(5), str(4)
memory usage: 84.5 KB


In [6]:
df.describe()

,store_id,is_weekend,is_festival,competition_density,items_sold
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,25.385833,0.280833,0.115833,4.930000,272.564167
std,14.275858,0.449594,0.320158,2.692485,60.142360
min,1.000000,0.000000,0.000000,1.000000,95.000000
25%,13.000000,0.000000,0.000000,3.000000,234.000000
50%,25.000000,0.000000,0.000000,5.000000,271.000000
75%,37.000000,1.000000,0.000000,7.000000,310.000000
max,50.000000,1.000000,1.000000,9.000000,468.000000


In [7]:
df.isnull().sum()

transaction_date       0
store_id               0
store_size             0
location_type          0
promotion_type         0
is_weekend             0
is_festival            0
competition_density    0
items_sold             0
dtype: int64

Date Feature Engineering —  From transaction_date, extract: year, month, day_of_week. Create an additional binary feature is_month_end (set to 1 if day of month ≥ 25, else 0). Display a sample of the resulting dataframe to confirm the new columns.

In [8]:
## convert the date from string to object as we want extract details

df['transaction_date'] = pd.to_datetime(df['transaction_date'])


In [9]:
df

,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249
3,2022-01-02,17,small,urban,free_gift,1,0,7,259
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277
...,...,...,...,...,...,...,...,...,...
1195,2024-12-28,39,large,urban,bogo,1,0,3,384
1196,2024-12-28,44,medium,urban,category_offer,1,0,9,371
1197,2024-12-29,47,large,semi-urban,bogo,1,0,7,367
1198,2024-12-31,25,small,urban,bogo,0,0,6,321


In [10]:
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek

In [11]:
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

In [12]:
df.head()

,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold,year,month,day_of_week,is_month_end
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224,2022,1,5,0
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348,2022,1,5,0
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249,2022,1,6,0
3,2022-01-02,17,small,urban,free_gift,1,0,7,259,2022,1,6,0
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277,2022,1,0,0


2. Temporal Train-Test Split — Sort the data by transaction_date. Use the most recent 20% of records as the test set and the remaining 80% as the training set. Do not use a random split. In a markdown cell, explain why a random split is inappropriate for time-ordered data.

In [13]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df = df.sort_values(by='transaction_date', ascending=False)

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression

In [15]:
categorical_features = ['promotion_type', 'location_type', 'store_size']
numerical_features = [
    'year', 'month', 'day_of_week', 'is_month_end', 
    'is_weekend', 'is_festival', 'competition_density'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

retail_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

4. Model Training and Evaluation — 8 marks